In [1]:
import pandas as pd 
import numpy as np 

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix , accuracy_score

In [2]:
data = pd.read_csv("C:\\Users\\bingi\\Downloads\\archive (1)\\bank-additional-full.csv")
df_clean = data.copy()
df_clean.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
df_clean.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [4]:
df_clean.drop(columns = 'default',axis = 1 , inplace = True)

In [5]:
cat_cols = df_clean.select_dtypes(include = 'object').columns

for col in cat_cols:
    print(f'{col}: {df_clean[col].unique()}')

job: ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']
marital: ['married' 'single' 'divorced' 'unknown']
education: ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']
housing: ['no' 'yes' 'unknown']
loan: ['no' 'yes' 'unknown']
contact: ['telephone' 'cellular']
month: ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']
day_of_week: ['mon' 'tue' 'wed' 'thu' 'fri']
poutcome: ['nonexistent' 'failure' 'success']
y: ['no' 'yes']


In [6]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
age,41188.0,40.024060,10.421250,17.000,32.000,38.000,47.000,98.000
duration,41188.0,258.285010,259.279249,0.000,102.000,180.000,319.000,4918.000
campaign,41188.0,2.567593,2.770014,1.000,1.000,2.000,3.000,56.000
pdays,41188.0,962.475454,186.910907,0.000,999.000,999.000,999.000,999.000
previous,41188.0,0.172963,0.494901,0.000,0.000,0.000,0.000,7.000
emp.var.rate,41188.0,0.081886,1.570960,-3.400,-1.800,1.100,1.400,1.400
cons.price.idx,41188.0,93.575664,0.578840,92.201,93.075,93.749,93.994,94.767
cons.conf.idx,41188.0,-40.502600,4.628198,-50.800,-42.700,-41.800,-36.400,-26.900
euribor3m,41188.0,3.621291,1.734447,0.634,1.344,4.857,4.961,5.045
nr.employed,41188.0,5167.035911,72.251528,4963.600,5099.100,5191.000,5228.100,5228.100


In [7]:
df_clean.columns

Index(['age', 'job', 'marital', 'education', 'housing', 'loan', 'contact',
       'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous',
       'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
       'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [8]:
df_clean.drop(columns = ['duration' , 'pdays'] , axis = 1 , inplace = True)

In [9]:
df_clean['y'].unique()

array(['no', 'yes'], dtype=object)

In [10]:
df_clean['y'] = df_clean['y'].map({'no' : 0 , 'yes' : 1})

In [11]:
df_clean['y'].value_counts()

y
0    36548
1     4640
Name: count, dtype: int64

In [12]:
y = df_clean['y']
X = df_clean.drop(columns = 'y' , axis = 1)

In [13]:
num_cols = X.select_dtypes(include = 'number').columns    
cat_cols = X.select_dtypes(exclude = 'number').columns

num_transformations = Pipeline(steps = [('imputer' , SimpleImputer(strategy = 'mean')),
                                        ('scaling' , StandardScaler())])

cat_transformations = Pipeline(steps = [('imputer' , SimpleImputer(strategy='most_frequent')),
                                        ('encoding' , OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num' , num_transformations , num_cols),
                                               ('cat' , cat_transformations , cat_cols)],
                                               remainder = 'passthrough')
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``.

In [14]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [15]:
X.shape

(41188, 17)

In [16]:
from sklearn.svm import LinearSVC
model = LinearSVC(C=1 ,class_weight='balanced', max_iter=5000)

final_pipeline = Pipeline([('preprocessor' , preprocessor) , ('model' , model)])
final_pipeline.fit(X_train , y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [17]:
y_pred = final_pipeline.predict(X_test)

print(classification_report(y_test , y_pred))
print(confusion_matrix(y_test , y_pred))
print(accuracy_score(y_test , y_pred))


              precision    recall  f1-score   support

           0       0.94      0.86      0.90      7303
           1       0.35      0.60      0.44       935

    accuracy                           0.83      8238
   macro avg       0.65      0.73      0.67      8238
weighted avg       0.88      0.83      0.85      8238

[[6245 1058]
 [ 370  565]]
0.8266569555717407


In [18]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# LinearSVC pipeline
svc_model = LinearSVC(C=1, class_weight='balanced', max_iter=5000)
svc_pipeline = Pipeline([('preprocessor', preprocessor), ('model', svc_model)])
svc_pipeline.fit(X_train, y_train)
y_pred_svc = svc_pipeline.predict(X_test)
print("LinearSVC Report:\n", classification_report(y_test, y_pred_svc))

# Logistic Regression pipeline
log_model = LogisticRegression(C=1, class_weight='balanced', max_iter=5000, solver='liblinear')
log_pipeline = Pipeline([('preprocessor', preprocessor), ('model', log_model)])
log_pipeline.fit(X_train, y_train)
y_pred_log = log_pipeline.predict(X_test)
print("Logistic Regression Report:\n", classification_report(y_test, y_pred_log))


LinearSVC Report:
               precision    recall  f1-score   support

           0       0.94      0.86      0.90      7303
           1       0.35      0.60      0.44       935

    accuracy                           0.83      8238
   macro avg       0.65      0.73      0.67      8238
weighted avg       0.88      0.83      0.85      8238

Logistic Regression Report:
               precision    recall  f1-score   support

           0       0.94      0.85      0.90      7303
           1       0.35      0.61      0.44       935

    accuracy                           0.83      8238
   macro avg       0.65      0.73      0.67      8238
weighted avg       0.88      0.83      0.85      8238



In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Define Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,        # number of trees
    max_depth=None,          # let trees grow fully
    class_weight='balanced', # handle imbalance
    random_state=42,         # reproducibility
    n_jobs=-1                # use all CPU cores
)

# Build pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

# Fit the pipeline
rf_pipeline.fit(X_train, y_train)


# Predict on test set
y_pred_rf = rf_pipeline.predict(X_test)

# Classification report
print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test , y_pred_rf))


Random Forest Report:
               precision    recall  f1-score   support

           0       0.91      0.97      0.94      7303
           1       0.54      0.29      0.38       935

    accuracy                           0.89      8238
   macro avg       0.73      0.63      0.66      8238
weighted avg       0.87      0.89      0.88      8238

Accuracy: 0.8909929594561787
[[7070  233]
 [ 665  270]]


In [20]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Calculate scale_pos_weight = (number of negatives / number of positives)
neg, pos = np.bincount(y_train)   # counts of each class in training set
scale_pos_weight = neg / pos

# Define XGBoost model
xgb_model = XGBClassifier(
    n_estimators=300,          # number of boosting rounds
    learning_rate=0.1,         # step size shrinkage
    max_depth=6,               # tree depth
    subsample=0.8,             # row sampling
    colsample_bytree=0.8,      # feature sampling
    scale_pos_weight=scale_pos_weight, # handle imbalance
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'      # avoids warning
)

# Build pipeline
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

# Fit the pipeline
xgb_pipeline.fit(X_train, y_train)

# Evaluate
y_pred_xgb = xgb_pipeline.predict(X_test)
print("XGBoost Report:\n", classification_report(y_test, y_pred_xgb))
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(confusion_matrix(y_test , y_pred_xgb))


XGBoost Report:
               precision    recall  f1-score   support

           0       0.94      0.87      0.90      7303
           1       0.36      0.56      0.44       935

    accuracy                           0.84      8238
   macro avg       0.65      0.72      0.67      8238
weighted avg       0.87      0.84      0.85      8238

Accuracy: 0.8373391599902889
[[6374  929]
 [ 411  524]]


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression' : LogisticRegression(
        C = 0.5,                    # smaller c value = strong penality
        solver = 'lbfgs',            # Optimization algorithm
        max_iter=100 ,             #no.of iterations
        class_weight= 'balanced'    #Adjust from imbalanced classes
        ),

    'Random Forest Classifier' : RandomForestClassifier(
        n_estimators=100,       # No.of trees
        max_depth=7,            # depth of tree
        min_samples_leaf= 2,     # minimum sample per leaf,
        min_samples_split = 4,   # minimum samples to split a node
        max_features='sqrt',     # Control randomness per split
        bootstrap=True,         # Sample with replacement
        random_state=42
    ),

    'Support Vector Machine' : SVC(
        C = 1.0,               # higher c value = less marigin and tighter fit
        kernel='rbf',     # for linear data (kernel = rbf , ploy , linear , sigmoid)
        gamma = 'scale',       # low gamma = smoother boundary
        probability=True,       #  Uses sigmoid to output probabilities
        class_weight='balanced',
        max_iter = 1000
    ),

    'KNeighbors Classifiers' : KNeighborsClassifier(
        n_neighbors = 3,         # no.of neighbors 
        weights='distance',      # Uniform or Distance
        metric = 'minkowski',    # Distance metric
        p = 2                   # p=2 (Euclidean) , p=1 (Manhattan)
    ),

    'Decision Tree' : DecisionTreeClassifier(
        max_depth = 5,          # Max depth
        min_samples_split = 4,  # min samples to split a node
        min_samples_leaf = 2,   # min samples of a leaf
        criterion = 'gini',     # Gini for measures impurity(less = more accurate) or entropy
        splitter = 'best',      # best vs random split
        class_weight = 'balanced'
    )
}

In [22]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Store all metrics together in a dictionary
    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "classification_report": classification_report(y_test, y_pred)
    }

# Print results neatly
for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    print("Accuracy:", metrics["accuracy"])
    print("Confusion Matrix:\n", metrics["confusion_matrix"])
    print("Classification Report:\n", metrics["classification_report"])


c:\Users\bingi\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\bingi\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\bingi\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\bingi\PycharmProjects\JupyterProject2\.v


Model: Logistic Regression
Accuracy: 0.8265355668851663
Confusion Matrix:
 [[6241 1062]
 [ 367  568]]
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.85      0.90      7303
           1       0.35      0.61      0.44       935

    accuracy                           0.83      8238
   macro avg       0.65      0.73      0.67      8238
weighted avg       0.88      0.83      0.85      8238


Model: Random Forest Classifier
Accuracy: 0.8971837824714737
Confusion Matrix:
 [[7215   88]
 [ 759  176]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.99      0.94      7303
           1       0.67      0.19      0.29       935

    accuracy                           0.90      8238
   macro avg       0.79      0.59      0.62      8238
weighted avg       0.88      0.90      0.87      8238


Model: Support Vector Machine
Accuracy: 0.11349842194707453
Confusion Matrix:
 [[   0 

In [28]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Loop through each model
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    # Predictions on train and test
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    # --- Train metrics ---
    print(f"\nModel: {name} (Train Data)")
    print("Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))
    print("Classification Report:\n", classification_report(y_train, y_train_pred, zero_division=0))
    
    # --- Test metrics ---
    print(f"\nModel: {name} (Test Data)")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    print("Classification Report:\n", classification_report(y_test, y_test_pred, zero_division=0))



Model: Logistic Regression (Train Data)
Accuracy: 0.8317754172989378
Confusion Matrix:
 [[25073  4172]
 [ 1371  2334]]
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.86      0.90     29245
           1       0.36      0.63      0.46      3705

    accuracy                           0.83     32950
   macro avg       0.65      0.74      0.68     32950
weighted avg       0.88      0.83      0.85     32950


Model: Logistic Regression (Test Data)
Accuracy: 0.8265355668851663
Confusion Matrix:
 [[6241 1062]
 [ 367  568]]
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.85      0.90      7303
           1       0.35      0.61      0.44       935

    accuracy                           0.83      8238
   macro avg       0.65      0.73      0.67      8238
weighted avg       0.88      0.83      0.85      8238


Model: Random Forest Classifier (Train Data)
Accuracy: 0.90333

c:\Users\bingi\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(



Model: Support Vector Machine (Train Data)
Accuracy: 0.11244309559939301
Confusion Matrix:
 [[    0 29245]
 [    0  3705]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00     29245
           1       0.11      1.00      0.20      3705

    accuracy                           0.11     32950
   macro avg       0.06      0.50      0.10     32950
weighted avg       0.01      0.11      0.02     32950


Model: Support Vector Machine (Test Data)
Accuracy: 0.11349842194707453
Confusion Matrix:
 [[   0 7303]
 [   0  935]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00      7303
           1       0.11      1.00      0.20       935

    accuracy                           0.11      8238
   macro avg       0.06      0.50      0.10      8238
weighted avg       0.01      0.11      0.02      8238


Model: KNeighbors Classifiers (Train Data)
Accuracy: 0